# engine

> Fill in a module description here

In [ ]:
# | default_exp engine

In [ ]:
# | export
import math
import os
import re
import socket
import subprocess
import threading
import time
from concurrent import futures
from pathlib import Path
from typing import Any, Callable, Generator

import grpc
import networkx as nx
import numpy as np
from google.protobuf import empty_pb2
from google.protobuf.json_format import MessageToDict

from spannerflow.config import Config
from spannerflow.dataflow.v1 import dataflow_pb2, dataflow_pb2_grpc
from spannerflow.graph_utils import find_output
from spannerflow.grpc_server import FunctionService
from spannerflow.span import Span

In [ ]:
# | export
def deserialize_span(span_str: str) -> Span:
    SPAN_PATTERN = r"^\[@(?P<doc_id>.*),(?P<start>\d+),(?P<end>\d+)\) \"(?P<text>.*)\"$"
    matches = re.match(SPAN_PATTERN, span_str)
    if matches is None:
        raise ValueError(f"Invalid span string: {span_str}")
    res = matches.groupdict()
    return Span(res["text"], int(res["start"]), int(res["end"]), res["doc_id"])


def deserialize_row(schema: list[str], row: list[str]) -> list[Any]:
    new_row: list[Any] = list()
    for col_type, value in zip(schema, row):
        match dataflow_pb2.DataType.Value(col_type):  # type: ignore
            case dataflow_pb2.DataType.DATA_TYPE_STRING:  # type: ignore
                new_row.append(value)  # alread a string
            case dataflow_pb2.DataType.DATA_TYPE_INT:  # type: ignore
                new_row.append(int(value))
            case dataflow_pb2.DataType.DATA_TYPE_FLOAT:  # type: ignore
                if math.isnan(float(value)):
                    raise ValueError(f"Expected float, got {value}")
                new_row.append(float(value))
            case dataflow_pb2.DataType.DATA_TYPE_BOOL:  # type: ignore
                if value not in ["true", "false"]:
                    raise ValueError(f"Expected bool, got {value}")
                new_row.append(value.lower() == "true")
            case dataflow_pb2.DataType.DATA_TYPE_SPAN:  # type: ignore
                new_row.append(deserialize_span(value))
            case dataflow_pb2.DataType.DATA_TYPE_INT64:  # type: ignore
                new_row.append(np.int64(value))
            case _:
                raise ValueError(f"Unknown data type: {col_type}")
    return new_row


def serialize_row(schema: list[str], row: list[Any]) -> list[str]:
    new_row = list()
    for col_type, value in zip(schema, row):
        match dataflow_pb2.DataType.Value(col_type):  # type: ignore
            case dataflow_pb2.DataType.DATA_TYPE_STRING:  # type: ignore
                if not isinstance(value, str):
                    raise ValueError(f"Expected str, got {type(value)}")
                new_row.append(value)
            case dataflow_pb2.DataType.DATA_TYPE_INT:  # type: ignore
                if not isinstance(value, int):
                    raise ValueError(f"Expected int, got {type(value)}")
                new_row.append(str(value))
            case dataflow_pb2.DataType.DATA_TYPE_FLOAT:  # type: ignore
                if not isinstance(value, (float, int)) or math.isnan(value):
                    raise ValueError(f"Expected float/int, got {value} ({type(value)})")
                new_row.append(str(value))
            case dataflow_pb2.DataType.DATA_TYPE_BOOL:  # type: ignore
                if not isinstance(value, bool):
                    raise ValueError(f"Expected bool, got {type(value)}")
                new_row.append(str(value).lower())
            case dataflow_pb2.DataType.DATA_TYPE_SPAN:  # type: ignore
                if isinstance(value, str):
                    value = Span(value)
                if not isinstance(value, Span):
                    raise ValueError(f"Expected Span, got {type(value)}")
                new_row.append(value.serialize())
            case dataflow_pb2.DataType.DATA_TYPE_INT64:  # type: ignore
                if not isinstance(value, (int, np.int64)):
                    raise ValueError(f"Expected int/numpy.int64, got {type(value)}")
                new_row.append(str(value))
            case _:
                raise ValueError(f"Unknown data type: {col_type}")
    return new_row

# Class Engine

In [ ]:
# | export


class Engine:
    def __init__(
        self,
        ie_functions: dict[
            str, tuple[str, Callable[[Any], Any], list[type], list[type]]
        ] = dict(),
        agg_functions: dict[
            str, tuple[str, Callable[[Any], Any], list[type], list[type]]
        ] = dict(),
    ):
        from spannerflow.rust_dataflow import RustDataflow

        self._ie_functions = ie_functions
        self._agg_functions = agg_functions
        self._is_rust_server_running = False
        self._is_python_server_running = False
        self._rust_server_process: None | subprocess.Popen = None
        self._python_grpc_server: None | grpc.server = None

        port_1, port_2 = self.find_server_ports()
        self._config = Config(DATAFLOW_PORT=port_1, LISTEN_PORT=port_2)
        self._rust_dataflow = RustDataflow(config=self._config, engine=self)  # type: ignore
        self._run_rust_server_in_background()
        self._run_python_server_in_background()
        time.sleep(1)

    def find_server_ports(self) -> tuple[int, int]:
        port_1 = None
        port_2 = None
        current_port = 50051
        while port_1 is None or port_2 is None:
            if self._is_port_available(current_port):
                if port_1 is None:
                    port_1 = current_port
                else:
                    port_2 = current_port
            current_port += 1
        return port_1, port_2

    def _is_port_available(self, port: int) -> bool:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            return sock.connect_ex(("localhost", port)) != 0

    def close(self):
        self._stop_rust_server()
        self._stop_python_server()

    def _run_python_server_in_background(self) -> None:
        def run_server() -> None:
            self._python_grpc_server = grpc.server(
                futures.ThreadPoolExecutor(max_workers=10)
            )
            dataflow_pb2_grpc.add_FunctionServiceServicer_to_server(
                FunctionService(self._ie_functions, self._agg_functions),
                self._python_grpc_server,
            )
            self._python_grpc_server.add_insecure_port(self._config.LISTEN_ADDRESS)
            self._python_grpc_server.start()
            self._python_grpc_server.wait_for_termination()

        if not self._is_python_server_running:
            threading.Thread(target=run_server, daemon=True).start()
            self._is_python_server_running = True

    def _stop_python_server(self):
        if not self._is_python_server_running:
            return
        self._python_grpc_server.stop(0)
        self._is_python_server_running = False

    def _run_rust_server_in_background(self) -> None:
        def inner() -> None:
            server_path = self._config.RUST_SERVER_PATH
            self._config.LOGS_DIR.mkdir(parents=True, exist_ok=True)
            with open(self._config.RUST_SERVER_LOG_PATH, "a") as log_file:
                env = os.environ.copy()
                env["BIND_PORT"] = str(self._config.DATAFLOW_PORT)
                self._rust_server_process = subprocess.Popen(
                    [str(server_path)],
                    stdout=log_file,
                    stderr=log_file,
                    env=env,
                )
                self._rust_server_process.communicate()

        if self._is_rust_server_running:
            return
        threading.Thread(target=inner).start()
        self._is_rust_server_running = True

    def _stop_rust_server(self):
        if not self._is_rust_server_running:
            return
        self._rust_server_process.terminate()
        self._is_rust_server_running = False

    def save_to_csv(
        self, collection_name: str, file_path: Path, delimiter: str = ","
    ) -> None:
        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request = dataflow_pb2.SaveToCSVRequest(  # type: ignore
                collection_name=collection_name,
                file_path=str(file_path),
                delimiter=delimiter.encode(),
            )
            stub.SaveToCSV(request)

    def load_from_csv(
        self,
        collection_name: str,
        file_path: Path,
        delimiter: str = ",",
        has_header: bool = False,
    ) -> None:
        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request = dataflow_pb2.LoadFromCSVRequest(  # type: ignore
                collection_name=collection_name,
                file_path=str(file_path),
                delimiter=delimiter.encode(),
                has_header=has_header,
            )
            stub.LoadFromCSV(request)

    def add_row(self, collection_name: str, row: list[Any]) -> None:
        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request = dataflow_pb2.AddRowRequest(  # type: ignore
                collection_name=collection_name,
                row=serialize_row(self.get_collections()[collection_name], row),
            )
            stub.AddRow(request)

    def _generate_add_rows_requests(
        self, collection_name: str, schema: list[str], rows: list[list[Any]]
    ) -> Generator[dataflow_pb2.AddRowsRequest, None, None]:  # type: ignore
        yield dataflow_pb2.AddRowsRequest(  # type: ignore
            collection_name=collection_name
        )
        for row in rows:
            yield dataflow_pb2.AddRowsRequest(  # type: ignore
                row=dataflow_pb2.RowRequest(row=serialize_row(schema, row))  # type: ignore
            )

    def add_rows(self, collection_name: str, rows: list[list[Any]]) -> None:
        schema = self.get_collections()[collection_name]
        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request_generator = self._generate_add_rows_requests(
                collection_name, schema, rows
            )
            stub.AddRows(request_generator)

    def delete_row(self, collection_name: str, row: list[Any]) -> None:
        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request = dataflow_pb2.DeleteRowRequest(  # type: ignore
                collection_name=collection_name,
                row=serialize_row(self.get_collections()[collection_name], row),
            )
            stub.DeleteRow(request)

    def add_collection(
        self,
        collection_name: str,
        schema: list[dataflow_pb2.DataType],  # type: ignore
    ) -> None:
        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request = dataflow_pb2.AddCollectionRequest(  # type: ignore
                collection_name=collection_name, schema=schema
            )
            stub.AddCollection(request)

    def delete_collection(self, collection_name: str) -> None:
        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request = dataflow_pb2.DeleteCollectionRequest(  # type: ignore
                collection_name=collection_name
            )
            stub.DeleteCollection(request)

    def get_collections(self) -> dict[str, list[str]]:
        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request = empty_pb2.Empty()
            response = stub.GetCollections(request)
            if "collections" not in MessageToDict(response):
                return dict()
            return {
                d["name"]: d["schema"] for d in MessageToDict(response)["collections"]
            }

    def get_collection(self, collection_name) -> Generator[list[str], None, None]:
        schema = self.get_collections()[collection_name]
        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request = dataflow_pb2.GetCollectionRequest(collection_name=collection_name)  # type: ignore
            response_iterator = stub.GetCollection(request)

            for response in response_iterator:
                yield deserialize_row(schema, response.row)

    def set_ie_function(
        self,
        name: str,
        func: Callable[[Any], Any],
        input_types: list[type],
        output_types: list[type],
    ):
        self._ie_functions[name] = (name, func, input_types, output_types)

    def get_ie_function(self, name: str):
        return self._ie_functions[name]

    def del_ie_function(self, name: str):
        del self._ie_functions[name]

    def set_agg_function(
        self,
        name: str,
        func: Callable[[Any], Any],
        input_types: list[type],
        output_types: list[type],
    ):
        self._agg_functions[name] = (name, func, input_types, output_types)

    def get_agg_function(self, name: str):
        return self._agg_functions[name]

    def del_agg_function(self, name: str):
        del self._agg_functions[name]

    def run_dataflow(
        self, reversed_graph: nx.DiGraph, output_csv_path: str | None = None
    ) -> Generator[list[str], None, None]:
        so_path, fn_name = self._rust_dataflow.build_so(reversed_graph)

        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request = dataflow_pb2.RunDataflowRequest(  # type: ignore
                so_path=str(so_path),
                fn_name=fn_name,
            )
            if output_csv_path is not None:
                request.output_csv_path = output_csv_path
            response_iterator = stub.RunDataflow(request)
            schema_types = reversed_graph.graph["nodes_schema_types_dict"][
                find_output(reversed_graph)
            ]
            for response in response_iterator:
                yield deserialize_row(
                    schema_types, [str(item) for item in response.row]
                )

    def get_span(self, doc_id: str, start: int, end: int) -> Span:
        with grpc.insecure_channel(self._config.DATAFLOW_ADDRESS) as channel:
            stub = dataflow_pb2_grpc.DataflowServiceStub(channel)
            request = dataflow_pb2.GetSpanRequest(id=doc_id, start=start, end=end)  # type: ignore
            response = stub.GetSpan(request)
            return Span(doc=response.span)

In [ ]:
"""
def get_span(docid, start, end):
    e = get_current_engine() // get the current engine from singleton
    e.get_span(docid, start, end)
"""

'\ndef get_span(docid, start, end):\n    e = get_current_engine() // get the current engine from singleton\n    e.get_span(docid, start, end)\n'

In [ ]:
engine = Engine(ie_functions={})

engine.add_collection(
    "X",
    [
        dataflow_pb2.DATA_TYPE_STRING,  # type: ignore
        dataflow_pb2.DATA_TYPE_INT,  # type: ignore
        dataflow_pb2.DATA_TYPE_BOOL,  # type: ignore
    ],
)

assert list(engine.get_collections()) == ["X"]


engine.add_row("X", ["a", 12, False])
engine.add_row("X", ["b", 10, True])
engine.add_rows(
    "X",
    [
        ["c", 10, False],
        ["d", 12, True],
    ],
)


assert list(engine.get_collection("X")) == [
    ["a", 12, False],
    ["b", 10, True],
    ["c", 10, False],
    ["d", 12, True],
]


engine.delete_row("X", ["a", 12, False])

assert list(engine.get_collection("X")) == [
    ["b", 10, True],
    ["c", 10, False],
    ["d", 12, True],
]

engine.delete_collection("X")

assert list(engine.get_collections()) == []

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()

ValueError: Unexpected format in '/Users/itarazi/Documents/Technion/spanner/spannerflow/spannerflow/rust_lib/.env/lib/python3.12/site-packages/nbdev/sync.py' at cell:
```
# %% ../nbs/api/06_sync.ipynb
from .imports import *
from .config import *
from .maker import *
from .process import *
from .process import _partition_cell
from .export import *
from .doclinks import _iter_py_cells

from execnb.nbio import *
from fastcore.script import *
from fastcore.xtras import *

import ast
import importlib.
```
The expected format is: '# %% {nb_path} {cell_idx}'.